In [ ]:
import numpy as np


def generate_and_fit_polynomial(
    total_points=1580,
    desired_fit_degree=4,
    periodicity=24,
    noise_std=0.05,
    seed=42,
    num_context=20,
    skip=700,
):
    np.random.seed(seed)

    def ground_truth_polynomial(x_scaled):
        # Higher-degree nonlinear structure
        p1 = 0.3 * x_scaled**4
        p2 = -0.2 * x_scaled**5
        p3 = 0.1 * x_scaled**6
        periodic = 0.3 * np.sin(2 * np.pi * x_scaled * periodicity)

        # Combine components
        values = p1 + p2 + p3 + periodic
        values -= values.min()
        values /= values.max()
        return values

    # --- FIX 1: Scale x to [-1, 1] ---
    x_raw = np.linspace(500, 2000, total_points)
    x_vals = np.linspace(
        -1, 1, total_points
    )  # fit over [-1, 1] for numerical stability

    # --- FIX 2: Generate target y ---
    y_clean = ground_truth_polynomial(x_vals)
    y_vals = y_clean + np.random.normal(0, noise_std, size=total_points)

    coeffs_and_errors = []
    y_true_vals = []
    # --- Fit and evaluate ---
    for i in range(skip, skip + num_context, 1):
        x_obs = x_vals[:i]
        y_obs = y_vals[:i]

        coeffs = np.polyfit(x_obs, y_obs, desired_fit_degree)
        poly = np.poly1d(coeffs)

        x_next = x_vals[i]
        y_pred = poly(x_next)
        y_true = y_vals[i]

        error = y_pred - y_true
        coeffs_and_errors.append(np.concatenate([coeffs, [error]]))
        y_true_vals.append(y_true)

    return x_raw, y_vals, np.array(coeffs_and_errors), np.array(y_true_vals)


# This forward is very slow, aboyt 1.5 hours for railway dataset
# context_fit = batch['context']['x'].squeeze(0)
# target_fit = batch['target']['x'].squeeze(0)

# context = batch['context']['y'].squeeze(0)
# target = batch['target']['y']
# outputs = []
# for idx in range(target.shape[1]):
#     # Get the right context and target for the current index

#     self.fit(context_fit[idx::24,:,], target_fit[idx::24,:,])
#     # if self.aux_context==None:
#     #     desired_fit_degree = 4
#     #     _, _, context_aux, y_aux = generate_and_fit_polynomial(desired_fit_degree=desired_fit_degree)

#     #     # Generate zeros
#     #     context_aux = np.concatenate([np.zeros((context_aux.shape[0], context_fit.shape[1] - desired_fit_degree - 2)), context_aux], axis=1)
#     #     self.aux_context = torch.Tensor(context_aux).to(batch['context']['x'].device)
#     #     self.y_aux = torch.Tensor(y_aux.reshape(-1,1)).to(batch['context']['x'].device)


#     # context_fit = torch.cat([self.aux_context, context_fit], dim=0)
#     # target_fit = torch.cat([self.y_aux, target_fit], dim=0)


#     output = self.backbone.predict(context[idx].unsqueeze(0).cpu())
#     outputs.append(output.reshape(-1, 1))

# outputs = np.concatenate(outputs, axis=0)#     outputs.append(output.reshape(-1, 1))

# outputs = np.concatenate(outputs, axis=0)

In [11]:
# Loader pytorch lightning chckpoint 
import torch
checkpoint = '/home/levtel/projects/TabPHM/checkpoints/autoencoders/battery/epoch004-val_loss0.00.ckpt'


# Use the context manager to temporarily allow this class

checkpoint = torch.load(checkpoint, weights_only=True)

print("Checkpoint loaded successfully and securely!")



UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL lmetk.loss.default.MSELoss was not an allowed global by default. Please use `torch.serialization.add_safe_globals([lmetk.loss.default.MSELoss])` or the `torch.serialization.safe_globals([lmetk.loss.default.MSELoss])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more abou

In [60]:
from lmetk.loss.default import MSELoss
import torch
checkpoint_path = '/home/levtel/projects/TabPHM/checkpoints/autoencoders/battery/epoch004-val_loss0.00-v1.ckpt'

# Define a list of classes you trust
safe_globals = [MSELoss]

# Use the context manager to temporarily allow this class during loading
with torch.serialization.safe_globals(safe_globals):
    checkpoint = torch.load(checkpoint_path, weights_only=False)

print("Checkpoint loaded successfully and securely!")

Checkpoint loaded successfully and securely!


In [62]:
checkpoint.keys() #['hyper_parameters']

dict_keys(['epoch', 'global_step', 'pytorch-lightning_version', 'state_dict', 'loops', 'hparams_name', 'hyper_parameters', 'datamodule_hparams_name', 'datamodule_hyper_parameters'])

In [39]:
checkpoint['datamodule_hparams_name']